In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier



df = pd.read_csv("autonomous_driving_expanded_dataset.csv")


le = LabelEncoder()
df['behavior_label'] = le.fit_transform(df['behavior_label'])

X = df.drop('behavior_label', axis=1)
y = df['behavior_label']

# 3. Custom Transformer Function 
def car_data_transformer(df_input):
    X_copy = df_input.copy()
    
    # Unit Conversion
    if 'speed_limit_kmh' in X_copy.columns:
        X_copy['speed_limit_mps'] = X_copy['speed_limit_kmh'] / 3.6
        X_copy = X_copy.drop(columns=['speed_limit_kmh'])
    
    # Categorical Mapping
    road_mapping = {'dry': 3, 'wet': 2, 'icy': 1}
    if 'road_surface_condition' in X_copy.columns:
        X_copy['road_surface_condition'] = X_copy['road_surface_condition'].map(road_mapping).fillna(3)
        
    return X_copy

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = [
    'obstacle_distance_m', 'relative_speed_mps', 'num_obstacles', 
    'lane_offset_m', 'traffic_density_veh_per_km', 'risk_probability', 
    'road_curvature_1pm', 'road_width_m', 'speed_limit_mps', 
    'ego_speed_mps', 'ego_acceleration_mps2', 'steering_angle_deg', 
    'yaw_rate_rads', 'throttle_position', 'brake_pressure', 
    'visibility_range_m', 'road_surface_condition'
]
cat_cols = ['weather_condition']


num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

rf_model3 = RandomForestClassifier(n_estimators= 200, min_samples_split= 10, max_depth= 30, bootstrap= False, class_weight='balanced', random_state=42)


autoDriving_pipeline = Pipeline(steps=[
    ('custom_logic', FunctionTransformer(car_data_transformer)),
    ('preprocessor', preprocessor),
    ('classifier', rf_model3) 
])

st_kfolds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_val_score(autoDriving_pipeline, X, y, cv=st_kfolds)



autoDriving_pipeline.fit(X, y)

autoDriving_pipeline.target_names = list(le.classes_)

joblib.dump(autoDriving_pipeline, 'autoDriving_pipeline.pkl')

print("Pipeline saved successfully")

Pipeline saved successfully
